# 🧠 DeepFake Detector - Entrenamiento con GPU

**Este notebook SOLO entrena.** El dataset ya está organizado en tu Google Drive.

### 📋 Pasos:
1. Conectar GPU y Google Drive
2. Instalar dependencias
3. Verificar que las imágenes existen
4. Entrenar (~20-40 min con GPU T4)
5. Descargar modelo

---

## 🔧 Paso 1: Verificar GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memoria: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("❌ NO hay GPU. Ve a Runtime > Change runtime type > GPU")
    assert False, "Necesitas GPU para entrenar"

## 🔗 Paso 2: Conectar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive conectado!")

## 📦 Paso 3: Instalar dependencias

In [ ]:
!pip install -q torch torchvision pillow opencv-python-headless numpy matplotlib seaborn scikit-learn tqdm
print("✅ Dependencias listas!")

## ⚙️ Paso 4: Configurar proyecto

In [ ]:
import sys, os
from pathlib import Path

PROJECT_DIR = '/content/drive/MyDrive/detectorIA'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

print(f"Proyecto: {PROJECT_DIR}")
print(f"Contenido: {os.listdir(PROJECT_DIR)}")

## 🔍 Paso 5: Verificar que las imágenes existen

In [ ]:
from pathlib import Path

DATA_DIR = Path(PROJECT_DIR) / 'data'

print("📊 Verificando imágenes en Drive:")
print(f"   Ruta: {DATA_DIR}")
print()

total = 0
for split in ['train', 'validation', 'test']:
    for cls in ['real', 'fake']:
        d = DATA_DIR / split / cls
        if d.exists():
            count = len(list(d.glob('*')))
            total += count
            print(f"  ✅ {split}/{cls}: {count} imágenes")
        else:
            print(f"  ❌ {split}/{cls}: NO EXISTE ({d})")

print(f"\n  Total: {total} imágenes")

if total == 0:
    print("\n❌ No se encontraron imágenes. Verifica que subiste el dataset a Google Drive.")
    print(f"   Estructura esperada:")
    print(f"   {DATA_DIR}/train/real/*.jpg")
    print(f"   {DATA_DIR}/train/fake/*.jpg")
    print(f"   {DATA_DIR}/validation/real/*.jpg")
    print(f"   {DATA_DIR}/validation/fake/*.jpg")
    print(f"   {DATA_DIR}/test/real/*.jpg")
    print(f"   {DATA_DIR}/test/fake/*.jpg")
    assert False, "No hay imágenes para entrenar"
else:
    print("\n✅ ¡Imágenes listas para entrenar!")

## 🚀 Paso 6: Entrenar el modelo

Con GPU T4, esto tardará **~20-40 minutos** para 30 épocas.

In [ ]:
from src.train import train

print(f"🎮 Dispositivo: GPU ({torch.cuda.get_device_name(0)})")
print(f"⏱️  Épocas: 30 | Batch: 32 | Tiempo estimado: 20-40 min")
print("=" * 50)
print()

history, test_metrics = train(
    epochs=30,
    batch_size=32,
    num_workers=2,
    freeze_until_block=3
)

print("\n" + "=" * 50)
print("  ✅ ENTRENAMIENTO COMPLETADO!")
print("=" * 50)
print(f"  Test Accuracy: {test_metrics['metrics']['accuracy']:.4f}")
print(f"  Test AUC:      {test_metrics['metrics']['auc']:.4f}")
print(f"  Épocas:        {history['epochs_trained']}")

## 📊 Paso 7: Ver resultados

In [ ]:
from IPython.display import Image, display
import json

plots_dir = os.path.join(PROJECT_DIR, 'outputs', 'plots')

for img_name in ['training_history.png', 'confusion_matrix.png', 'roc_curve.png']:
    img_path = os.path.join(plots_dir, img_name)
    if os.path.exists(img_path):
        print(f"\n--- {img_name} ---")
        display(Image(filename=img_path, width=600))

metrics_path = os.path.join(plots_dir, 'final_metrics.json')
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        m = json.load(f)
    print("\n📋 Métricas finales:")
    for k, v in m.items():
        if isinstance(v, float):
            print(f"  {k:<20} {v:.4f}")

## 💾 Paso 8: Descargar modelo

El modelo se guardó automáticamente en tu Google Drive:
```
Mi unidad/detectorIA/outputs/checkpoints/
├── best_model.pth
└── last_model.pth
```

**Para usarlo en tu computadora:**
1. Ve a Google Drive > detectorIA > outputs > checkpoints
2. Descarga `best_model.pth`
3. Pégalo en tu carpeta local: `detectorIA/outputs/checkpoints/`

In [ ]:
ck_dir = os.path.join(PROJECT_DIR, 'outputs', 'checkpoints')
print("💾 Checkpoints en Drive:")
if os.path.exists(ck_dir):
    for f in os.listdir(ck_dir):
        mb = os.path.getsize(os.path.join(ck_dir, f)) / 1024**2
        print(f"  📦 {f} ({mb:.1f} MB)")

print("\n✅ ¡Listo! Descarga best_model.pth a tu computadora.")